In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from google.colab import drive
import statsmodels.api as sm
import numpy as np


In [ ]:
file_path = "/content/drive/MyDrive/Analisis matemático/FINAL 2027 CSV.csv"
df_filtrado = pd.read_csv(file_path, encoding='latin1')

In [ ]:
df_filtrado["POBLACION_TOTAL_CENTRADO"] = df_filtrado["POBLACION_TOTAL"] - df_filtrado.groupby("YEAR")["POBLACION_TOTAL"].transform("mean")
df_filtrado["POBLACION_0_14_CENTRADO"] = df_filtrado["POBLACION_0_14"] - df_filtrado.groupby("YEAR")["POBLACION_0_14"].transform("mean")

In [ ]:
# Crear una función para buscar el valor de mortalidad de 2020 o 2016
def imputar_mortalidad(row):
    if pd.isna(row["TASA_MORTALIDAD_INFANTIL_CENTRADO"]) and row["YEAR"] == 2024:
        # Buscar el país en 2020
        valor_2020 = df_filtrado[(df_filtrado["COUNTRY"] == row["COUNTRY"]) & (df_filtrado["YEAR"] == 2020)]["TASA_MORTALIDAD_INFANTIL_CENTRADO"]
        if not valor_2020.empty:
            return valor_2020.values[0]
        else:
            # Buscar el país en 2016
            valor_2016 = df_filtrado[(df_filtrado["COUNTRY"] == row["COUNTRY"]) & (df_filtrado["YEAR"] == 2016)]["TASA_MORTALIDAD_INFANTIL_CENTRADO"]
            if not valor_2016.empty:
                return valor_2016.values[0]
            else:
                # Si no hay 2020 ni 2016, imputar 0
                return 0
    else:
        return row["TASA_MORTALIDAD_INFANTIL_CENTRADO"]

# Aplicar la imputación
df_filtrado["TASA_MORTALIDAD_INFANTIL_CENTRADO"] = df_filtrado.apply(imputar_mortalidad, axis=1)
######################
def imputar_mortalidad2(row):
    if pd.isna(row["TASA_MORTALIDAD_INFANTIL"]) and row["YEAR"] == 2024:
        # Buscar el país en 2020
        valor_2020 = df_filtrado[(df_filtrado["COUNTRY"] == row["COUNTRY"]) & (df_filtrado["YEAR"] == 2020)]["TASA_MORTALIDAD_INFANTIL"]
        if not valor_2020.empty:
            return valor_2020.values[0]
        else:
            # Buscar el país en 2016
            valor_2016 = df_filtrado[(df_filtrado["COUNTRY"] == row["COUNTRY"]) & (df_filtrado["YEAR"] == 2016)]["TASA_MORTALIDAD_INFANTIL"]
            if not valor_2016.empty:
                return valor_2016.values[0]
            else:
                # Si no hay 2020 ni 2016, imputar 0
                return 0
    else:
        return row["TASA_MORTALIDAD_INFANTIL"]

# Aplicar la imputación
df_filtrado["TASA_MORTALIDAD_INFANTIL"] = df_filtrado.apply(imputar_mortalidad2, axis=1)

In [ ]:
# Impute missing values for all relevant columns based on group-wise mean
for column in ["POBLACION_0_14","PIB_REAL","POBLACION_TOTAL","POBLACION_20_24","LOG_PIB_REAL", "POBLACION_20_24_CENTRADO", "TASA_MORTALIDAD_INFANTIL_CENTRADO","TASA_MORTALIDAD_INFANTIL","POBLACION_TOTAL_CENTRADO","POBLACION_0_14_CENTRADO"]:
    # Group by 'YEAR' and fill NaN with the mean of the group
    df_filtrado[column] = df_filtrado.groupby("YEAR")[column].transform(lambda x: x.fillna(x.mean()))

In [ ]:
# Check for NaNs in all relevant columns after imputation
for column in ["POBLACION_0_14","PIB_REAL","POBLACION_TOTAL","POBLACION_20_24","LOG_PIB_REAL", "POBLACION_20_24_CENTRADO", "TASA_MORTALIDAD_INFANTIL_CENTRADO","TASA_MORTALIDAD_INFANTIL","POBLACION_TOTAL_CENTRADO","POBLACION_0_14_CENTRADO"]:
    print(f"NaNs in {column}: {df_filtrado[column].isna().sum()}")

NaNs in POBLACION_0_14: 0
NaNs in PIB_REAL: 0
NaNs in POBLACION_TOTAL: 0
NaNs in POBLACION_20_24: 0
NaNs in LOG_PIB_REAL: 0
NaNs in POBLACION_20_24_CENTRADO: 0
NaNs in TASA_MORTALIDAD_INFANTIL_CENTRADO: 0
NaNs in TASA_MORTALIDAD_INFANTIL: 0
NaNs in POBLACION_TOTAL_CENTRADO: 0
NaNs in POBLACION_0_14_CENTRADO: 0


In [ ]:
df_filtrado.to_excel('SINNANS.xlsx', index=False)

In [ ]:
# Definir X e y ahora que todo está limpio
# 1. Filtrar países que ganaron al menos una medalla (para evitar log(0))
df_modelo1 = df_filtrado[df_filtrado["PROP_MEDALLAS"] > 0].copy()
X = df_modelo1[[
    "POBLACION_TOTAL_CENTRADO",
    "LOG_PIB_REAL",
    "POBLACION_20_24_CENTRADO",
    "TASA_MORTALIDAD_INFANTIL_CENTRADO",
    "LOCAL",
    "VECINO"
]]

# Agregar constante
X = sm.add_constant(X)

# Definir y
y = df_modelo1["LOG_PROP_MEDALLAS"]

# Confirmar shapes
print("Shape final de X:", X.shape)
print("Shape final de y:", y.shape)
X.head()


Shape final de X: (1059, 7)
Shape final de y: (1059,)


,const,POBLACION_TOTAL_CENTRADO,LOG_PIB_REAL,POBLACION_20_24_CENTRADO,TASA_MORTALIDAD_INFANTIL_CENTRADO,LOCAL,VECINO
12,1.0,-6.697765e+06,6.050083,-1.362870e+06,40.806218,0,0
13,1.0,-4.303140e+06,6.530443,-5.452886e+05,34.844041,0,0
33,1.0,-3.643240e+07,9.027067,-5.576904e+06,-12.092746,0,0
40,1.0,-1.879932e+06,8.883573,-2.289790e+05,10.795135,0,0
42,1.0,2.821860e+04,8.190184,3.144951e+05,-5.243523,0,0


In [ ]:
# Ajustar el modelo log-lineal
modelo = sm.OLS(y, X).fit()

# Mostrar resumen del modelo
print(modelo.summary())

                            OLS Regression Results                            
Dep. Variable:      LOG_PROP_MEDALLAS   R-squared:                       0.258
Model:                            OLS   Adj. R-squared:                  0.254
Method:                 Least Squares   F-statistic:                     60.93
Date:                Tue, 29 Apr 2025   Prob (F-statistic):           6.87e-65
Time:                        14:05:53   Log-Likelihood:                -1827.7
No. Observations:                1059   AIC:                             3669.
Df Residuals:                    1052   BIC:                             3704.
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                                        coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
const 

In [ ]:
# Definir X e y ahora que todo está limpio
# 1. Filtrar países que ganaron al menos una medalla (para evitar log(0))
df_modelo = df_filtrado[df_filtrado["PROP_MEDALLAS"] > 0].copy()
X = df_modelo[[
    "POBLACION_0_14_CENTRADO",
    "POBLACION_TOTAL_CENTRADO",
    "PIB_REAL",
    "POBLACION_20_24_CENTRADO",
    "TASA_MORTALIDAD_INFANTIL_CENTRADO",
    "LOCAL",
    "VECINO"
]]

# Agregar constante
X = sm.add_constant(X)

# Definir y
y = df_modelo["LOG_PROP_MEDALLAS"]

# Confirmar shapes
print("Shape final de X:", X.shape)
print("Shape final de y:", y.shape)
X.head()

Shape final de X: (1059, 8)
Shape final de y: (1059,)


,const,POBLACION_0_14_CENTRADO,POBLACION_TOTAL_CENTRADO,PIB_REAL,POBLACION_20_24_CENTRADO,TASA_MORTALIDAD_INFANTIL_CENTRADO,LOCAL,VECINO
12,1.0,3.859999e+06,-6.697765e+06,424.148042,-1.362870e+06,40.806218,0,0
13,1.0,5.123528e+06,-4.303140e+06,685.702247,-5.452886e+05,34.844041,0,0
33,1.0,-9.357079e+06,-3.643240e+07,8325.408868,-5.576904e+06,-12.092746,0,0
40,1.0,1.628789e+06,-1.879932e+06,7212.516326,-2.289790e+05,10.795135,0,0
42,1.0,2.203431e+06,2.821860e+04,3605.386016,3.144951e+05,-5.243523,0,0


In [ ]:
# Ajustar el modelo log-lineal
modelo = sm.OLS(y, X).fit()

# Mostrar resumen del modelo
print(modelo.summary())

                            OLS Regression Results                            
Dep. Variable:      LOG_PROP_MEDALLAS   R-squared:                       0.275
Model:                            OLS   Adj. R-squared:                  0.270
Method:                 Least Squares   F-statistic:                     56.88
Date:                Tue, 29 Apr 2025   Prob (F-statistic):           3.75e-69
Time:                        14:06:02   Log-Likelihood:                -1815.6
No. Observations:                1059   AIC:                             3647.
Df Residuals:                    1051   BIC:                             3687.
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                                        coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
const 

In [ ]:
import numpy as np

# 1. Predecir log(ŵ_it) con el modelo
log_predicciones = modelo.predict(X)

# 2. Destransformar: volver a escala real (ŵ_it)
predicciones = np.exp(log_predicciones)

# 3. Agregar las predicciones al dataframe de modelado
df_modelo["PROP_MEDALLAS_PREDICHA"] = predicciones

# 4. Opcional: calcular error absoluto entre proporción real y predicha
df_modelo["ERROR_ABSOLUTO"] = np.abs(df_modelo["PROP_MEDALLAS"] - df_modelo["PROP_MEDALLAS_PREDICHA"])

# 5. Mostrar ejemplo de tabla de resultados
resultados = df_modelo[[
    "COUNTRY",
    "YEAR",
    "MEDALLAS_TOTALES_POR_PAIS_POR_AÑO",
    "PROP_MEDALLAS",
    "PROP_MEDALLAS_PREDICHA",
    "ERROR_ABSOLUTO"
]].sort_values(by="ERROR_ABSOLUTO", ascending=False)

# Mostrar los 10 países con mayor error
resultados.head(10)


,COUNTRY,YEAR,MEDALLAS_TOTALES_POR_PAIS_POR_AÑO,PROP_MEDALLAS,PROP_MEDALLAS_PREDICHA,ERROR_ABSOLUTO
661,China,2020,149,0.065236,1.955617,1.890381
662,China,2024,172,0.075938,1.552477,1.476539
658,China,2008,184,0.090152,0.581409,0.491257
660,China,2016,113,0.056079,0.516369,0.460290
2606,Russia,1980,442,0.319134,0.054760,0.264374
2605,Russia,1976,286,0.216667,0.008142,0.208525
1160,Germany,1976,273,0.206818,0.010795,0.196023
2608,Russia,1988,300,0.189753,0.007701,0.182052
1161,Germany,1980,264,0.190614,0.012152,0.178462
2601,Russia,1960,169,0.185919,0.007832,0.178086


In [ ]:
# 1. Primero, recuperar el total de medallas entregadas cada año
total_medallas_por_ano = df_modelo.groupby("YEAR")["MEDALLAS_TOTALES_POR_PAIS_POR_AÑO"].transform("sum")

# 2. Calcular las medallas reales y predichas en números absolutos
df_modelo["MEDALLAS_REALES"] = df_modelo["PROP_MEDALLAS"] * total_medallas_por_ano
df_modelo["MEDALLAS_PREDICHAS"] = df_modelo["PROP_MEDALLAS_PREDICHA"] * total_medallas_por_ano

# 3. Calcular error absoluto en número de medallas
df_modelo["ERROR_ABSOLUTO_MEDALLAS"] = np.abs(df_modelo["MEDALLAS_REALES"] - df_modelo["MEDALLAS_PREDICHAS"])

# 4. Armar la tabla ordenada
tabla_resultados = df_modelo[[
    "COUNTRY",
    "YEAR",
    "MEDALLAS_REALES",
    "MEDALLAS_PREDICHAS",
    "ERROR_ABSOLUTO_MEDALLAS"
]].sort_values(by="ERROR_ABSOLUTO_MEDALLAS", ascending=False)

# 5. Mostrar los 10 países con mayor error absoluto
tabla_resultados.head(10)


,COUNTRY,YEAR,MEDALLAS_REALES,MEDALLAS_PREDICHAS,ERROR_ABSOLUTO_MEDALLAS
661,China,2020,148.999999,4466.629675,4317.629676
662,China,2024,172.000000,3516.359975,3344.359975
658,China,2008,183.999999,1186.654912,1002.654913
660,China,2016,112.999999,1040.483470,927.483471
2606,Russia,1980,442.000000,75.842184,366.157816
2608,Russia,1988,300.000001,12.175668,287.824332
3344,United States,2008,317.000001,37.632265,279.367736
1163,Germany,1988,295.999999,17.324893,278.675107
2605,Russia,1976,286.000000,10.747175,275.252825
1160,Germany,1976,273.000000,14.249076,258.750925


In [ ]:
tabla_resultados.to_excel('RESULTADOS2.xlsx', index=False)

In [ ]:
# Definir X e y ahora que todo está limpio
# 1. Filtrar países que ganaron al menos una medalla (para evitar log(0))
df_modelo = df_filtrado[df_filtrado["PROP_MEDALLAS"] > 0].copy()
X = df_modelo[[
    "POBLACION_0_14",
    "POBLACION_TOTAL",
    "PIB_REAL",
    "POBLACION_20_24",
    "TASA_MORTALIDAD_INFANTIL",
    "LOCAL",
    "VECINO"
]]

# Agregar constante
X = sm.add_constant(X)

# Definir y
y = df_modelo["LOG_PROP_MEDALLAS"]

# Confirmar shapes
print("Shape final de X:", X.shape)
print("Shape final de y:", y.shape)
X.head()

Shape final de X: (1059, 8)
Shape final de y: (1059,)


,const,POBLACION_0_14,POBLACION_TOTAL,PIB_REAL,POBLACION_20_24,TASA_MORTALIDAD_INFANTIL,LOCAL,VECINO
12,1.0,13043065,26482622.0,424.148042,4457025.283,69.6,0,0
13,1.0,14548198,30560034.0,685.702247,5485526.103,60.1,0,0
33,1.0,467968,2745972.0,8325.408868,389928.024,8.4,0,0
40,1.0,9580584,21271969.0,7212.516326,3941695.856,67.7,0,0
42,1.0,10910578,26628568.0,3605.386016,5187245.046,42.1,0,0


In [ ]:
# Ajustar el modelo log-lineal
modelo = sm.OLS(y, X).fit()

# Mostrar resumen del modelo
print(modelo.summary())

                            OLS Regression Results                            
Dep. Variable:      LOG_PROP_MEDALLAS   R-squared:                       0.200
Model:                            OLS   Adj. R-squared:                  0.194
Method:                 Least Squares   F-statistic:                     37.47
Date:                Tue, 29 Apr 2025   Prob (F-statistic):           5.02e-47
Time:                        14:06:23   Log-Likelihood:                -1867.7
No. Observations:                1059   AIC:                             3751.
Df Residuals:                    1051   BIC:                             3791.
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                               coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------
const                   

In [ ]:
import statsmodels.api as sm

# 1. Crear Dummies para las grandes potencias
df_filtrado["COUNTRY"] = df_filtrado["COUNTRY"].str.strip()

df_filtrado["DUMMY_CHINA"] = df_filtrado["COUNTRY"].apply(lambda x: 1 if x == "China" else 0)
df_filtrado["DUMMY_RUSSIA"] = df_filtrado["COUNTRY"].apply(lambda x: 1 if x in ["Russia", "Soviet Union", "USSR"] else 0)
df_filtrado["DUMMY_USA"] = df_filtrado["COUNTRY"].apply(lambda x: 1 if x == "United States" else 0)
df_filtrado["DUMMY_GERMANY"] = df_filtrado["COUNTRY"].apply(lambda x: 1 if x == "Germany" else 0)

# Definir X e y ahora que todo está limpio
# 1. Filtrar países que ganaron al menos una medalla (para evitar log(0))
df_modelo = df_filtrado[df_filtrado["PROP_MEDALLAS"] > 0].copy()
X = df_modelo[[
    "POBLACION_TOTAL_CENTRADO",
    "PIB_REAL",
    "POBLACION_20_24_CENTRADO",
    "TASA_MORTALIDAD_INFANTIL_CENTRADO",
    "LOCAL",
    "VECINO",
    "DUMMY_CHINA",
    "DUMMY_RUSSIA",
    "DUMMY_USA",
    "DUMMY_GERMANY"
]]

# Agregar constante
X = sm.add_constant(X)

# Definir y
y = df_modelo["LOG_PROP_MEDALLAS"]

# Confirmar shapes
print("Shape final de X:", X.shape)
print("Shape final de y:", y.shape)
X.head()

Shape final de X: (1059, 11)
Shape final de y: (1059,)


,const,POBLACION_TOTAL_CENTRADO,PIB_REAL,POBLACION_20_24_CENTRADO,TASA_MORTALIDAD_INFANTIL_CENTRADO,LOCAL,VECINO,DUMMY_CHINA,DUMMY_RUSSIA,DUMMY_USA,DUMMY_GERMANY
12,1.0,-6.697765e+06,424.148042,-1.362870e+06,40.806218,0,0,0,0,0,0
13,1.0,-4.303140e+06,685.702247,-5.452886e+05,34.844041,0,0,0,0,0,0
33,1.0,-3.643240e+07,8325.408868,-5.576904e+06,-12.092746,0,0,0,0,0,0
40,1.0,-1.879932e+06,7212.516326,-2.289790e+05,10.795135,0,0,0,0,0,0
42,1.0,2.821860e+04,3605.386016,3.144951e+05,-5.243523,0,0,0,0,0,0


In [ ]:
# Ajustar el modelo log-lineal
modelo = sm.OLS(y, X).fit()

# Mostrar resumen del modelo
print(modelo.summary())

                            OLS Regression Results                            
Dep. Variable:      LOG_PROP_MEDALLAS   R-squared:                       0.370
Model:                            OLS   Adj. R-squared:                  0.364
Method:                 Least Squares   F-statistic:                     61.43
Date:                Tue, 29 Apr 2025   Prob (F-statistic):           6.32e-98
Time:                        14:06:31   Log-Likelihood:                -1741.4
No. Observations:                1059   AIC:                             3505.
Df Residuals:                    1048   BIC:                             3559.
Df Model:                          10                                         
Covariance Type:            nonrobust                                         
                                        coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
const 

In [ ]:
import numpy as np

# 1. Predecir log(ŵ_it) con el modelo
log_predicciones = modelo.predict(X)

# 2. Destransformar: volver a escala real (ŵ_it)
predicciones = np.exp(log_predicciones)

# 3. Agregar las predicciones al dataframe de modelado
df_modelo["PROP_MEDALLAS_PREDICHA"] = predicciones

# 4. Opcional: calcular error absoluto entre proporción real y predicha
df_modelo["ERROR_ABSOLUTO"] = np.abs(df_modelo["PROP_MEDALLAS"] - df_modelo["PROP_MEDALLAS_PREDICHA"])

# 5. Mostrar ejemplo de tabla de resultados
resultados = df_modelo[[
    "COUNTRY",
    "YEAR",
    "MEDALLAS_TOTALES_POR_PAIS_POR_AÑO",
    "PROP_MEDALLAS",
    "PROP_MEDALLAS_PREDICHA",
    "ERROR_ABSOLUTO"
]].sort_values(by="ERROR_ABSOLUTO", ascending=False)

# Mostrar los 10 países con mayor error
resultados.head(10)


,COUNTRY,YEAR,MEDALLAS_TOTALES_POR_PAIS_POR_AÑO,PROP_MEDALLAS,PROP_MEDALLAS_PREDICHA,ERROR_ABSOLUTO
3341,United States,1996,259,0.140684,0.594095,0.453410
661,China,2020,149,0.065236,0.420904,0.355667
662,China,2024,172,0.075938,0.363553,0.287615
1159,Germany,1972,253,0.208230,0.472793,0.264563
2606,Russia,1980,442,0.319134,0.570923,0.251789
3338,United States,1984,352,0.238644,0.489141,0.250497
1160,Germany,1976,273,0.206818,0.095950,0.110869
660,China,2016,113,0.056079,0.165558,0.109479
658,China,2008,184,0.090152,0.196811,0.106659
2605,Russia,1976,286,0.216667,0.120045,0.096622


In [ ]:
# 1. Primero, recuperar el total de medallas entregadas cada año
total_medallas_por_ano = df_modelo.groupby("YEAR")["MEDALLAS_TOTALES_POR_PAIS_POR_AÑO"].transform("sum")

# 2. Calcular las medallas reales y predichas en números absolutos
df_modelo["MEDALLAS_REALES"] = df_modelo["PROP_MEDALLAS"] * total_medallas_por_ano
df_modelo["MEDALLAS_PREDICHAS"] = df_modelo["PROP_MEDALLAS_PREDICHA"] * total_medallas_por_ano

# 3. Calcular error absoluto en número de medallas
df_modelo["ERROR_ABSOLUTO_MEDALLAS"] = np.abs(df_modelo["MEDALLAS_REALES"] - df_modelo["MEDALLAS_PREDICHAS"])

# 4. Armar la tabla ordenada
tabla_resultados = df_modelo[[
    "COUNTRY",
    "YEAR",
    "MEDALLAS_REALES",
    "MEDALLAS_PREDICHAS",
    "ERROR_ABSOLUTO_MEDALLAS"
]].sort_values(by="ERROR_ABSOLUTO_MEDALLAS", ascending=False)

# 5. Mostrar los 10 países con mayor error absoluto
tabla_resultados.head(15)


,COUNTRY,YEAR,MEDALLAS_REALES,MEDALLAS_PREDICHAS,ERROR_ABSOLUTO_MEDALLAS
3341,United States,1996,259.000001,1093.728233,834.728233
661,China,2020,148.999999,961.344440,812.344441
662,China,2024,172.000000,823.447129,651.447128
3338,United States,1984,352.000000,721.483333,369.483333
2606,Russia,1980,442.000000,790.727938,348.727938
1159,Germany,1972,253.000000,574.443439,321.443439
660,China,2016,112.999999,333.599270,220.599271
658,China,2008,183.999999,401.691025,217.691025
1160,Germany,1976,273.000000,126.653449,146.346551
164,Australia,2004,157.000000,11.434349,145.565651


In [ ]:
# 1. Crear dummies para boicots y potencias modernas

# Dummies de boicots
df_modelo["BOICOT_1980"] = df_modelo["YEAR"].apply(lambda x: 1 if x == 1980 else 0)
df_modelo["BOICOT_1984"] = df_modelo["YEAR"].apply(lambda x: 1 if x == 1984 else 0)

# Dummy Australia
df_modelo["DUMMY_AUSTRALIA"] = df_modelo["COUNTRY"].apply(lambda x: 1 if x == "Australia" else 0)

# Dummy United Kingdom
df_modelo["DUMMY_UK"] = df_modelo["COUNTRY"].apply(lambda x: 1 if x in ["United Kingdom", "Great Britain"] else 0)

# 2. Redefinir X incluyendo todas las nuevas variables
X = df_modelo[[
    "POBLACION_TOTAL_CENTRADO",
    "PIB_REAL",
    "POBLACION_20_24_CENTRADO",
    "TASA_MORTALIDAD_INFANTIL_CENTRADO",
    "LOCAL",
    "VECINO",
    "DUMMY_CHINA",
    "DUMMY_RUSSIA",
    "DUMMY_USA",
    "DUMMY_GERMANY",
    #"BOICOT_1980",
    #"BOICOT_1984",
    "DUMMY_AUSTRALIA",
    "DUMMY_UK"
]]

# 3. Agregar constante
X = sm.add_constant(X)

# 4. Definir y
y = df_modelo["LOG_PROP_MEDALLAS"]

# 5. Ajustar el nuevo modelo
modelo = sm.OLS(y, X).fit()

# 6. Mostrar resumen del nuevo modelo
print(modelo.summary())


                            OLS Regression Results                            
Dep. Variable:      LOG_PROP_MEDALLAS   R-squared:                       0.404
Model:                            OLS   Adj. R-squared:                  0.397
Method:                 Least Squares   F-statistic:                     58.99
Date:                Tue, 29 Apr 2025   Prob (F-statistic):          1.48e-108
Time:                        14:06:45   Log-Likelihood:                -1712.0
No. Observations:                1059   AIC:                             3450.
Df Residuals:                    1046   BIC:                             3515.
Df Model:                          12                                         
Covariance Type:            nonrobust                                         
                                        coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
const 

In [ ]:
import numpy as np

# 1. Predecir log(ŵ_it) con el modelo
log_predicciones = modelo.predict(X)

# 2. Destransformar: volver a escala real (ŵ_it)
predicciones = np.exp(log_predicciones)

# 3. Agregar las predicciones al dataframe de modelado
df_modelo["PROP_MEDALLAS_PREDICHA"] = predicciones

# 4. Opcional: calcular error absoluto entre proporción real y predicha
df_modelo["ERROR_ABSOLUTO"] = np.abs(df_modelo["PROP_MEDALLAS"] - df_modelo["PROP_MEDALLAS_PREDICHA"])

# 5. Mostrar ejemplo de tabla de resultados
resultados = df_modelo[[
    "COUNTRY",
    "YEAR",
    "MEDALLAS_TOTALES_POR_PAIS_POR_AÑO",
    "PROP_MEDALLAS",
    "PROP_MEDALLAS_PREDICHA",
    "ERROR_ABSOLUTO"
]].sort_values(by="ERROR_ABSOLUTO", ascending=False)

# Mostrar los 10 países con mayor error
resultados.head(10)


,COUNTRY,YEAR,MEDALLAS_TOTALES_POR_PAIS_POR_AÑO,PROP_MEDALLAS,PROP_MEDALLAS_PREDICHA,ERROR_ABSOLUTO
3341,United States,1996,259,0.140684,0.522619,0.381935
661,China,2020,149,0.065236,0.400049,0.334813
662,China,2024,172,0.075938,0.317855,0.241917
1159,Germany,1972,253,0.208230,0.411335,0.203104
3338,United States,1984,352,0.238644,0.433750,0.195106
2606,Russia,1980,442,0.319134,0.493288,0.174154
1160,Germany,1976,273,0.206818,0.094485,0.112334
2605,Russia,1976,286,0.216667,0.118240,0.098427
660,China,2016,113,0.056079,0.152220,0.096141
1163,Germany,1988,296,0.187223,0.092308,0.094915


In [ ]:
# 1. Primero, recuperar el total de medallas entregadas cada año
total_medallas_por_ano = df_modelo.groupby("YEAR")["MEDALLAS_TOTALES_POR_PAIS_POR_AÑO"].transform("sum")

# 2. Calcular las medallas reales y predichas en números absolutos
df_modelo["MEDALLAS_REALES"] = df_modelo["PROP_MEDALLAS"] * total_medallas_por_ano
df_modelo["MEDALLAS_PREDICHAS"] = df_modelo["PROP_MEDALLAS_PREDICHA"] * total_medallas_por_ano

# 3. Calcular error absoluto en número de medallas
df_modelo["ERROR_ABSOLUTO_MEDALLAS"] = np.abs(df_modelo["MEDALLAS_REALES"] - df_modelo["MEDALLAS_PREDICHAS"])

# 4. Armar la tabla ordenada
tabla_resultados = df_modelo[[
    "COUNTRY",
    "YEAR",
    "MEDALLAS_REALES",
    "MEDALLAS_PREDICHAS",
    "ERROR_ABSOLUTO_MEDALLAS"
]].sort_values(by="ERROR_ABSOLUTO_MEDALLAS", ascending=False)

# 5. Mostrar los 10 países con mayor error absoluto
tabla_resultados.head(15)


,COUNTRY,YEAR,MEDALLAS_REALES,MEDALLAS_PREDICHAS,ERROR_ABSOLUTO_MEDALLAS
661,China,2020,148.999999,913.712628,764.712628
3341,United States,1996,259.000001,962.142134,703.142133
662,China,2024,172.000000,719.941419,547.941419
3338,United States,1984,352.000000,639.781068,287.781068
1159,Germany,1972,253.000000,499.771508,246.771507
2606,Russia,1980,442.000000,683.203614,241.203614
660,China,2016,112.999999,306.724115,193.724116
658,China,2008,183.999999,354.102391,170.102391
1163,Germany,1988,295.999999,145.938972,150.061027
1160,Germany,1976,273.000000,124.719665,148.280335


In [ ]:
tabla_resultados.to_excel('RESULTADOS3.xlsx', index=False)

In [ ]:
# 1. Crear nuevos dummies

# China moderna (año >= 2000)
df_modelo["DUMMY_CHINA_MODERNA"] = df_modelo.apply(lambda row: 1 if row["COUNTRY"] == "China" and row["YEAR"] >= 2008 else 0, axis=1)

# Francia (en general)
df_modelo["DUMMY_FRANCE"] = df_modelo["COUNTRY"].apply(lambda x: 1 if x == "France" else 0)

# 2. Redefinir X con las nuevas variables
X = df_modelo[[
    "PIB_REAL",
    "POBLACION_TOTAL_CENTRADO",
    "POBLACION_20_24_CENTRADO",
    "TASA_MORTALIDAD_INFANTIL_CENTRADO",
    "LOCAL",
    "VECINO",
    "DUMMY_CHINA_MODERNA",
    "DUMMY_RUSSIA",
    "DUMMY_USA",
    "DUMMY_GERMANY",
    "BOICOT_1980",
    "BOICOT_1984",
    "DUMMY_AUSTRALIA",
    "DUMMY_UK",
    "DUMMY_FRANCE"
]]

# 3. Agregar constante
X = sm.add_constant(X)

# 4. Definir y
y= df_modelo["LOG_PROP_MEDALLAS"]

# 5. Ajustar el modelo
modelo= sm.OLS(y, X).fit()

# 6. Mostrar el resumen del modelo
print(modelo.summary())


                            OLS Regression Results                            
Dep. Variable:      LOG_PROP_MEDALLAS   R-squared:                       0.417
Model:                            OLS   Adj. R-squared:                  0.409
Method:                 Least Squares   F-statistic:                     49.73
Date:                Tue, 29 Apr 2025   Prob (F-statistic):          5.67e-111
Time:                        14:34:25   Log-Likelihood:                -1700.0
No. Observations:                1059   AIC:                             3432.
Df Residuals:                    1043   BIC:                             3511.
Df Model:                          15                                         
Covariance Type:            nonrobust                                         
                                        coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
const 

In [ ]:
df_modelo.to_excel('FINALMODELO.xlsx', index=False)

In [ ]:
import numpy as np

# 1. Predecir log(ŵ_it) con el modelo
log_predicciones = modelo.predict(X)

# 2. Destransformar: volver a escala real (ŵ_it)
predicciones = np.exp(log_predicciones)

# 3. Agregar las predicciones al dataframe de modelado
df_modelo["PROP_MEDALLAS_PREDICHA"] = predicciones

# 4. Opcional: calcular error absoluto entre proporción real y predicha
df_modelo["ERROR_ABSOLUTO"] = np.abs(df_modelo["PROP_MEDALLAS"] - df_modelo["PROP_MEDALLAS_PREDICHA"])

# 5. Mostrar ejemplo de tabla de resultados
resultados = df_modelo[[
    "COUNTRY",
    "YEAR",
    "MEDALLAS_TOTALES_POR_PAIS_POR_AÑO",
    "PROP_MEDALLAS",
    "PROP_MEDALLAS_PREDICHA",
    "ERROR_ABSOLUTO"
]].sort_values(by="ERROR_ABSOLUTO", ascending=False)

# Mostrar los 10 países con mayor error
resultados.head(10)


,COUNTRY,YEAR,MEDALLAS_TOTALES_POR_PAIS_POR_AÑO,PROP_MEDALLAS,PROP_MEDALLAS_PREDICHA,ERROR_ABSOLUTO
3341,United States,1996,259,0.140684,0.540193,0.399508
2606,Russia,1980,442,0.319134,0.589997,0.270863
3338,United States,1984,352,0.238644,0.440224,0.201580
1159,Germany,1972,253,0.208230,0.406963,0.198732
661,China,2020,149,0.065236,0.228489,0.163252
662,China,2024,172,0.075938,0.210035,0.134097
1160,Germany,1976,273,0.206818,0.093610,0.113208
2605,Russia,1976,286,0.216667,0.112334,0.104332
1163,Germany,1988,296,0.187223,0.088382,0.098841
2608,Russia,1988,300,0.189753,0.115363,0.074390


In [ ]:
# 1. Primero, recuperar el total de medallas entregadas cada año
total_medallas_por_ano = df_modelo.groupby("YEAR")["MEDALLAS_TOTALES_POR_PAIS_POR_AÑO"].transform("sum")

# 2. Calcular las medallas reales y predichas en números absolutos
df_modelo["MEDALLAS_REALES"] = df_modelo["PROP_MEDALLAS"] * total_medallas_por_ano
df_modelo["MEDALLAS_PREDICHAS"] = df_modelo["PROP_MEDALLAS_PREDICHA"] * total_medallas_por_ano

# 3. Calcular error absoluto en número de medallas
df_modelo["ERROR_ABSOLUTO_MEDALLAS"] = np.abs(df_modelo["MEDALLAS_REALES"] - df_modelo["MEDALLAS_PREDICHAS"])

# 4. Armar la tabla ordenada
tabla_resultados = df_modelo[[
    "COUNTRY",
    "YEAR",
    "MEDALLAS_REALES",
    "MEDALLAS_PREDICHAS",
    "ERROR_ABSOLUTO_MEDALLAS"
]].sort_values(by="ERROR_ABSOLUTO_MEDALLAS", ascending=False)

# 5. Mostrar los 10 países con mayor error absoluto
tabla_resultados.head(15)


,COUNTRY,YEAR,MEDALLAS_REALES,MEDALLAS_PREDICHAS,ERROR_ABSOLUTO_MEDALLAS
3341,United States,1996,259.000001,994.495024,735.495024
2606,Russia,1980,442.000000,817.145550,375.145550
661,China,2020,148.999999,521.868437,372.868438
662,China,2024,172.000000,475.729561,303.729560
3338,United States,1984,352.000000,649.330400,297.330400
1159,Germany,1972,253.000000,494.459458,241.459457
1163,Germany,1988,295.999999,139.732238,156.267761
1160,Germany,1976,273.000000,123.565055,149.434945
2605,Russia,1976,286.000000,148.281383,137.718617
1172,Germany,2024,108.000000,231.048120,123.048120


In [ ]:
tabla_resultados.to_excel('RESULTADOS4.xlsx', index=False)